# Custom Markets with Custom Fleet

This notebook demonstrates how to define user-specified markets and a matching fleet.
Instead of the default four markets, we use two passenger markets and one freight market:

| Market id | Name | Type | RPK share 2019 | Energy share 2019 |
|-----------|------|------|:--------------:|:-----------------:|
| `regional` | Regional | passenger | 62.3 % | 52.19 % |
| `intercontinental` | Intercontinental | passenger | 37.7 % | 32.81 % |
| `air_cargo` | Air Cargo | freight | — | 15 % |

In [ ]:
from aeromaps import create_process

In [ ]:
process = create_process(configuration_file="data/config.yaml")
process.compute()

## Markets loaded

In [ ]:
for m in process.markets:
    print(f"{m.id:25s}  name={m.name!r:22s}  type={m.traffic_type}")

## Fleet categories

In [ ]:
for cat_name, cat in process.fleet_model.fleet.categories.items():
    subcats = [sc.name for sc in cat.subcategories.values()]
    print(f"[{cat.market_id}] {cat_name!r}  — subcategories: {subcats}")

## Per-market traffic

In [ ]:
vector = process.data["vector_outputs"]
years = [2019, 2030, 2050]

traffic_cols = ["rpk_regional", "rpk_intercontinental", "rtk_air_cargo"]
(vector[traffic_cols] / 1e9).loc[years]

## Global aggregates

In [ ]:
vector[["rpk", "ask", "load_factor", "co2_emissions_passenger", "co2_emissions_freight"]].loc[years]

## CO2 emissions par marché

In [ ]:
import matplotlib.pyplot as plt

vector = process.data["vector_outputs"]
market_ids = [m.id for m in process.markets]
co2_cols = [f"co2_emissions_{mid}" for mid in market_ids]

fig, ax = plt.subplots(figsize=(9, 4))
for col, mid in zip(co2_cols, market_ids):
    if col in vector.columns:
        ax.plot(vector.index, vector[col], label=mid)

ax.set_xlabel("Year")
ax.set_ylabel("CO₂ emissions (Mt)")
ax.set_title("CO₂ emissions par marché")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()